# PAC Kaggle Competition - Other Submission (B)

**INTRODUCTION**  
This notebook (Notebook B), provides a record of the exploratory path that informed the final approach. It documents alternative preprocessing decisions (including PCA-based dimensionality reduction), extensive model trials across linear, tree-based, and ensemble methods, and several missteps that ultimately guided the decision to favor a feature-engineered linear model. While many of these exploratory models performed reasonably well, none surpassed the engineered-feature workflow developed in Notebook A.

***Competition Overview:***  
A credit card company has provided historical data on its active customers. The data includes demographic and behavioral information on customers. Your task is to build a model to predict monthly credit card spend for individual customers.
 
Goal: Build a predictive model to estimate monthly credit card spending of individual customers based on a rich set of customer attributes, including demographics, credit behavior, transaction activity, and lifestyle indicators.


***Student Information:***  
- **Name:** Olivia (Chieh) Chou - UNI: cc5399
- **Course & Section:** APAN 5200 Section 007
- **Professor:** Andrew Assing
- **Kaggle Username:** ochou813  
- **Kaggle Display Name:** Olivia Chou


## Outline
End-to-end Workflow including:

1. Data Import & Inspection
2. Data Cleaning & Preprocessing
   * Set up Data
   * Dummy Coding
   * Handle Missing Values
   * Train/Test Split
   * PCA & Standard Scaler
3. Linear Models
   * Linear
   * Lasso
   * Ridge Regressions
4. Decision Tree Models
   * Simple Tree
   * Pruned Tree
5. Ensemble Methods
   * Voting Regressor
   * Random Forest
   * Bagging Regression
   * Adaboost
   * Gradient Boosting
6. Model Evaluation Summary
7. Predicting on Scoring Data with Best Model

In [1]:
# Import Packages & Helpers
# Keeping helpers at the top ensures consistency across all model comparisons.
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error

from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    BaggingRegressor,
    AdaBoostRegressor,
    GradientBoostingRegressor,
    VotingRegressor
)

RANDOM_STATE = 42

def rmse(y_true, y_pred):
    return root_mean_squared_error(y_true, y_pred)

## 1. Data Import & Initial Innspection
<hr>

In [2]:
analysis_data = pd.read_csv("analysis_data.csv")
scoring_data = pd.read_csv("scoring_data.csv")

target_col = "monthly_spend"
id_col = "customer_id"

print("Analysis shape:", analysis_data.shape)
print("Scoring shape:", scoring_data.shape)

Analysis shape: (40000, 23)
Scoring shape: (10000, 22)


In [3]:
analysis_data.head()

,customer_id,age,gender,marital_status,education_level,region,employment_status,owns_home,has_auto_loan,annual_income,...,card_type,num_transactions,avg_transaction_value,online_shopping_freq,reward_points_balance,travel_frequency,utility_payment_count,num_children,num_credit_cards,monthly_spend
0,75570383,77,female,married,high school,northeast,unemployed,1,0,56000.116893,...,standard,14,46.144967,8.0,6325.578928,3,3.0,1,8,1682.86
1,77915507,52,male,married,high school,south,unemployed,1,1,39432.589311,...,gold,13,88.532219,7.0,5586.814175,1,1.0,2,8,2135.57
2,31958910,48,female,married,bachelors,west,unemployed,1,1,51020.949673,...,standard,9,78.304445,4.0,6250.225115,3,2.0,0,8,1540.61
3,76868487,67,female,married,high school,northeast,unemployed,1,0,76794.566399,...,standard,10,54.932667,7.0,8875.409103,3,3.0,2,8,1250.86
4,69435006,21,female,single,bachelors,west,self-employed,1,1,44386.444631,...,standard,8,18.866712,6.0,5318.021523,4,3.0,0,8,915.36


In [4]:
analysis_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   customer_id            40000 non-null  int64  
 1   age                    40000 non-null  int64  
 2   gender                 40000 non-null  object 
 3   marital_status         40000 non-null  object 
 4   education_level        38801 non-null  object 
 5   region                 40000 non-null  object 
 6   employment_status      40000 non-null  object 
 7   owns_home              40000 non-null  int64  
 8   has_auto_loan          40000 non-null  int64  
 9   annual_income          40000 non-null  float64
 10  credit_score           40000 non-null  float64
 11  credit_limit           40000 non-null  float64
 12  tenure                 40000 non-null  int64  
 13  card_type              40000 non-null  object 
 14  num_transactions       40000 non-null  int64  
 15  av

In [5]:
scoring_data.head()

,customer_id,age,gender,marital_status,education_level,region,employment_status,owns_home,has_auto_loan,annual_income,...,tenure,card_type,num_transactions,avg_transaction_value,online_shopping_freq,reward_points_balance,travel_frequency,utility_payment_count,num_children,num_credit_cards
0,20451981,68,female,married,bachelors,west,self-employed,1,0,28736.958529,...,17,standard,6,32.444032,6.0,4360.785803,3,5.0,2,8
1,23251656,37,male,married,high school,northeast,student,0,1,177014.762457,...,14,platinum,17,30.655429,18.0,18790.889746,7,7.0,3,8
2,64082260,65,female,married,bachelors,northeast,self-employed,0,0,51625.522861,...,1,standard,9,36.063881,5.0,5625.669555,5,1.0,1,8
3,82505923,33,female,married,bachelors,northeast,self-employed,0,1,62667.564975,...,2,gold,12,28.833727,11.0,6752.725039,1,4.0,3,8
4,89981994,23,male,single,bachelors,midwest,self-employed,1,0,84996.492293,...,7,gold,15,12.361809,7.0,9154.342800,6,0.0,0,8


In [6]:
scoring_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   customer_id            10000 non-null  int64  
 1   age                    10000 non-null  int64  
 2   gender                 10000 non-null  object 
 3   marital_status         10000 non-null  object 
 4   education_level        9699 non-null   object 
 5   region                 10000 non-null  object 
 6   employment_status      10000 non-null  object 
 7   owns_home              10000 non-null  int64  
 8   has_auto_loan          10000 non-null  int64  
 9   annual_income          10000 non-null  float64
 10  credit_score           10000 non-null  float64
 11  credit_limit           10000 non-null  float64
 12  tenure                 10000 non-null  int64  
 13  card_type              10000 non-null  object 
 14  num_transactions       10000 non-null  int64  
 15  avg

## 2. Data Cleaning & Preprocessing
<hr>

- Impute missing values  
- Apply dummy encoding  
- Split into training/testing sets
- Feature Selection wth PCA

**Goal**

Prepare the dataset for PCA and for consistent comparison across many modeling methods. Preprocessing for Notebook B is intentionally simpler than Notebook A, because the goal is experimental breadth, not optimal performance.

### 2.1 Set Up Data

In [7]:
# Separate target
y = analysis_data[target_col].values
analysis_features = analysis_data.drop(columns=[target_col])

# Combine for uniform preprocessing
combined = pd.concat([analysis_features, scoring_data], ignore_index=True)

### 2.2 Dummy Coding

In [8]:
# Identify numeric & categorical columns
numeric_cols = combined.select_dtypes(include=["int64", "float64"]).columns.tolist()
if id_col in numeric_cols:
    numeric_cols.remove(id_col)

categorical_cols = combined.select_dtypes(include=["object"]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

Numeric columns: ['age', 'owns_home', 'has_auto_loan', 'annual_income', 'credit_score', 'credit_limit', 'tenure', 'num_transactions', 'avg_transaction_value', 'online_shopping_freq', 'reward_points_balance', 'travel_frequency', 'utility_payment_count', 'num_children', 'num_credit_cards']
Categorical columns: ['gender', 'marital_status', 'education_level', 'region', 'employment_status', 'card_type']


#### 2.2.1 Handle Missing Values

In [9]:
# Numeric: median imputation
for col in numeric_cols:
    combined[col] = combined[col].fillna(combined[col].median())

# Categorical: mode imputation
for col in categorical_cols:
    combined[col] = combined[col].fillna(combined[col].mode()[0])

In [10]:
# Dummy encoding
combined_encoded = pd.get_dummies(combined, columns=categorical_cols, drop_first=True)
print("Encoded combined shape:", combined_encoded.shape)


Encoded combined shape: (50000, 28)


In [11]:
# Split back
X_all = combined_encoded
X_analysis = X_all.iloc[:len(analysis_data), :].copy()
X_scoring = X_all.iloc[len(analysis_data):, :].copy()

### 2.3 Train/Test Split

In [12]:
# Remove ID column for training
X_analysis = X_analysis.drop(columns=[id_col])
X_scoring_model = X_scoring.drop(columns=[id_col])

In [13]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_analysis, y, test_size=0.2, random_state=RANDOM_STATE
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)

X_train: (32000, 27) X_test: (8000, 27)


### 2.4 Feature Selection/Extraction - PCA

**Purpose**

Use Principal Component Analysis to reduce dimensionality, remove redundancy, and extract latent signals.

**Insight**
- PCA was primarily exploratory; it helped evaluate how much variance is captured by correlated dummy and numeric features.
- The latent root criterion (eigenvalue ≥ 1) selected component count.
- PCA simplified the feature set but did not improve RMSE relative to feature-engineered models in Notebook A.
- Most behavioral and financial structure was lost when projecting into PCA space, leading to weaker performance.

#### 2.4.1 Standardization

In [14]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
X_scoring_scaled = scaler.transform(X_scoring_model)

#### 2.4.2 Correlaton Matrix

In [15]:
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns
)
corr_matrix = X_train_scaled_df.corr()
print("Correlation matrix (head):")
print(corr_matrix.head())

Correlation matrix (head):
                    age  owns_home  has_auto_loan  annual_income  \
age            1.000000   0.006769      -0.003813       0.004892   
owns_home      0.006769   1.000000       0.009119      -0.003543   
has_auto_loan -0.003813   0.009119       1.000000       0.008975   
annual_income  0.004892  -0.003543       0.008975       1.000000   
credit_score  -0.005630  -0.003353       0.000515      -0.003523   

               credit_score  credit_limit    tenure  num_transactions  \
age               -0.005630      0.005258  0.003649          0.002082   
owns_home         -0.003353     -0.002819  0.005616         -0.001925   
has_auto_loan      0.000515      0.008876  0.004864          0.005365   
annual_income     -0.003523      0.980668  0.002453          0.830782   
credit_score       1.000000      0.086267 -0.002848         -0.000526   

               avg_transaction_value  online_shopping_freq  ...  \
age                        -0.009193              0.007177

#### 2.4.3 PCA: 
Find eigenvalues & choose components with eigenvalue >= 1

In [16]:
pca_full = PCA()
pca_full.fit(X_train_scaled)
eigenvalues = pca_full.explained_variance_

print("First 10 eigenvalues:", eigenvalues[:10])

First 10 eigenvalues: [5.62296673 1.70845229 1.54354593 1.40028317 1.33794673 1.33370754
 1.33196408 1.2402198  1.01925987 1.00742066]


In [17]:
n_components = np.sum(eigenvalues >= 1.0)
if n_components == 0:
    # Fallback: at least 1 component
    n_components = 1

print("Selected number of components (eigenvalue >= 1):", n_components)

Selected number of components (eigenvalue >= 1): 11


In [18]:
# ---- Refit PCA using selected n_components
pca = PCA(n_components=n_components)
X_train_pc = pca.fit_transform(X_train_scaled)
X_test_pc  = pca.transform(X_test_scaled)
X_scoring_pc = pca.transform(X_scoring_scaled)  # for final prediction

print("X_train_pc:", X_train_pc.shape, "X_test_pc:", X_test_pc.shape)

X_train_pc: (32000, 11) X_test_pc: (8000, 11)


## 3. Linear Models
<hr>

**Purpose**  
Evaluate baseline linear models (OLS, Lasso, Ridge) in PCA-transformed space.

**Insight**
- Linear models performed similarly across the board.
- Regularization (Lasso, Ridge) offered no meaningful improvement.
- PCA + linear models did not outperform Notebook A’s engineered-feature linear regression.
- Confirmed that the strongest signal in this dataset is linear but not captured well by PCA.


In [19]:
results = []
best_models = {}

### 3.1 Linear Regression

In [20]:
lin_reg = LinearRegression()
param_grid_lin = {"fit_intercept": [True, False]}

grid_lin = GridSearchCV(
    lin_reg,
    param_grid_lin,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1
)
grid_lin.fit(X_train_pc, y_train)
best_lin = grid_lin.best_estimator_

y_train_pred = best_lin.predict(X_train_pc)
y_test_pred = best_lin.predict(X_test_pc)

results.append({
    "model": "LinearRegression",
    "train_rmse": rmse(y_train, y_train_pred),
    "test_rmse": rmse(y_test, y_test_pred)
})
best_models["LinearRegression"] = best_lin

print("Linear Regression best params:", grid_lin.best_params_)
print("Linear Regression RMSE - Train:", rmse(y_train, y_train_pred),
      "Test:", rmse(y_test, y_test_pred))

Linear Regression best params: {'fit_intercept': True}
Linear Regression RMSE - Train: 315.1020082734619 Test: 315.21477463966625


### 3.2 Lasso Regression

In [21]:
lasso = Lasso(max_iter=10000, random_state=RANDOM_STATE)
param_grid_lasso = {"alpha": [0.001, 0.01, 0.1, 1, 10]}

grid_lasso = GridSearchCV(
    lasso, param_grid_lasso, scoring="neg_root_mean_squared_error", cv=5, n_jobs=-1
)
grid_lasso.fit(X_train_pc, y_train)
best_lasso = grid_lasso.best_estimator_

results.append({
    "model": "Lasso",
    "train_rmse": rmse(y_train, best_lasso.predict(X_train_pc)),
    "test_rmse": rmse(y_test, best_lasso.predict(X_test_pc))
})
best_models["Lasso"] = best_lasso

print("Lasso best params:", grid_lasso.best_params_)
print("Lasso RMSE - Train:", results[-1]["train_rmse"],
      "Test:", results[-1]["test_rmse"])

Lasso best params: {'alpha': 0.1}
Lasso RMSE - Train: 315.1021376319334 Test: 315.21681560571903


### 3.3 Ridge Regression

In [22]:
ridge = Ridge(max_iter=10000, random_state=RANDOM_STATE)
param_grid_ridge = {"alpha": [0.001, 0.01, 0.1, 1, 10]}

grid_ridge = GridSearchCV(
    ridge, param_grid_ridge, scoring="neg_root_mean_squared_error", cv=5, n_jobs=-1
)
grid_ridge.fit(X_train_pc, y_train)
best_ridge = grid_ridge.best_estimator_

results.append({
    "model": "Ridge",
    "train_rmse": rmse(y_train, best_ridge.predict(X_train_pc)),
    "test_rmse": rmse(y_test, best_ridge.predict(X_test_pc))
})
best_models["Ridge"] = best_ridge

print("Ridge best params:", grid_ridge.best_params_)
print("Ridge RMSE - Train:", results[-1]["train_rmse"],
      "Test:", results[-1]["test_rmse"])

Ridge best params: {'alpha': 10}
Ridge RMSE - Train: 315.1020127934871 Test: 315.21543620941094


## 4. Decision Tree Models
<hr>

**Purpose:**  
Explore non-linear modeling through simple and pruned decision trees.

**Insight:**  
- Simple tree underfitted severely.  
- Pruned tree improved but still far worse than linear models.  
- Trees revealed that non-linear splits did not capture spending behavior effectively.

### 4.1 Simple Tree (max_depth = 2)

In [23]:
tree_simple = DecisionTreeRegressor(max_depth=2, random_state=RANDOM_STATE)
tree_simple.fit(X_train_pc, y_train)

results.append({
    "model": "DecisionTree_simple",
    "train_rmse": rmse(y_train, tree_simple.predict(X_train_pc)),
    "test_rmse": rmse(y_test, tree_simple.predict(X_test_pc))
})
best_models["DecisionTree_simple"] = tree_simple

print("Simple Tree RMSE - Train:", results[-1]["train_rmse"],
      "Test:", results[-1]["test_rmse"])

Simple Tree RMSE - Train: 403.5499322147385 Test: 407.8743760342058


### 4.2 Pruned Tree (with gridsearch)

In [24]:
tree = DecisionTreeRegressor(random_state=RANDOM_STATE)
param_grid_tree = {
    "max_depth": [2,3,4,5,6,7,8,9,10],
    "min_samples_leaf": [1,5,10,15,20],
    "min_samples_split": [2,10,20,30]
}

grid_tree = GridSearchCV(
    tree, param_grid_tree, scoring="neg_root_mean_squared_error", cv=5, n_jobs=-1
)
grid_tree.fit(X_train_pc, y_train)
best_tree = grid_tree.best_estimator_

results.append({
    "model": "DecisionTree_pruned",
    "train_rmse": rmse(y_train, best_tree.predict(X_train_pc)),
    "test_rmse": rmse(y_test, best_tree.predict(X_test_pc))
})
best_models["DecisionTree_pruned"] = best_tree

print("Pruned Tree best params:", grid_tree.best_params_)
print("Pruned Tree RMSE - Train:", results[-1]["train_rmse"],
      "Test:", results[-1]["test_rmse"])

Pruned Tree best params: {'max_depth': 10, 'min_samples_leaf': 20, 'min_samples_split': 2}
Pruned Tree RMSE - Train: 298.4797923797381 Test: 332.38320935708384


## 5. Ensemble Methods
<hr>

**Purpose**  
Explore advanced nonlinear models capable of capturing complex patterns missed by linear approaches.

**Insight**
- Random Forest: extremely low train RMSE but higher test RMSE → strong overfitting
- Bagging: more stable but still weaker than linear models
- AdaBoost: improved over simple trees but still suboptimal
- Gradient Boosting: best among ensembles, but still inferior to engineered linear models

Overall, ensembles handled noise better than trees but worse than OLS in PCA space. This confirmed that PCA removed meaningful structure needed for ensemble models to succeed.

### 5.1 Voting Regressor

In [25]:
voting_reg = VotingRegressor([
    ("lin", best_lin),
    ("tree", best_tree)
])
voting_reg.fit(X_train_pc, y_train)

results.append({
    "model": "VotingRegressor",
    "train_rmse": rmse(y_train, voting_reg.predict(X_train_pc)),
    "test_rmse": rmse(y_test, voting_reg.predict(X_test_pc))
})
best_models["VotingRegressor"] = voting_reg

print("Voting Regressor RMSE - Train:", results[-1]["train_rmse"],
      "Test:", results[-1]["test_rmse"])

Voting Regressor RMSE - Train: 293.1591217879862 Test: 309.9607329650509


### 5.2 Random Forest

In [26]:
rf = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)
param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth": [None, 5, 10],
    "max_features": ["sqrt", "log2"]
}

grid_rf = GridSearchCV(
    rf, param_grid_rf, scoring="neg_root_mean_squared_error", cv=5, n_jobs=-1
)
grid_rf.fit(X_train_pc, y_train)
best_rf = grid_rf.best_estimator_

results.append({
    "model": "RandomForest",
    "train_rmse": rmse(y_train, best_rf.predict(X_train_pc)),
    "test_rmse": rmse(y_test, best_rf.predict(X_test_pc))
})
best_models["RandomForest"] = best_rf

print("Random Forest best params:", grid_rf.best_params_)
print("Random Forest RMSE - Train:", results[-1]["train_rmse"],
      "Test:", results[-1]["test_rmse"])

/opt/anaconda3/envs/appliedML/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Random Forest best params: {'max_depth': None, 'max_features': 'sqrt', 'n_estimators': 200}
Random Forest RMSE - Train: 108.47077251037116 Test: 292.1125950887992


### 5.3 Bagging Regressor

In [27]:
bag_base = DecisionTreeRegressor(max_depth=5, random_state=RANDOM_STATE)
bag = BaggingRegressor(estimator=bag_base, random_state=RANDOM_STATE, n_jobs=-1)

param_grid_bag = {
    "n_estimators": [50, 100],
    "max_samples": [0.8, 1.0]
}

grid_bag = GridSearchCV(
    bag, param_grid_bag, scoring="neg_root_mean_squared_error", cv=5, n_jobs=-1
)
grid_bag.fit(X_train_pc, y_train)
best_bag = grid_bag.best_estimator_

results.append({
    "model": "Bagging",
    "train_rmse": rmse(y_train, best_bag.predict(X_train_pc)),
    "test_rmse": rmse(y_test, best_bag.predict(X_test_pc))
})
best_models["Bagging"] = best_bag

print("Bagging best params:", grid_bag.best_params_)
print("Bagging RMSE - Train:", results[-1]["train_rmse"],
      "Test:", results[-1]["test_rmse"])

Bagging best params: {'max_samples': 0.8, 'n_estimators': 100}
Bagging RMSE - Train: 334.3153974535784 Test: 337.5533601682182


### 5.4 Adaboost

In [28]:
ada_base = DecisionTreeRegressor(max_depth=2, random_state=RANDOM_STATE)
ada = AdaBoostRegressor(estimator=ada_base, random_state=RANDOM_STATE)

param_grid_ada = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1.0]
}

grid_ada = GridSearchCV(
    ada, param_grid_ada, scoring="neg_root_mean_squared_error", cv=5, n_jobs=-1
)
grid_ada.fit(X_train_pc, y_train)
best_ada = grid_ada.best_estimator_

results.append({
    "model": "AdaBoost",
    "train_rmse": rmse(y_train, best_ada.predict(X_train_pc)),
    "test_rmse": rmse(y_test, best_ada.predict(X_test_pc))
})
best_models["AdaBoost"] = best_ada

print("AdaBoost best params:", grid_ada.best_params_)
print("AdaBoost RMSE - Train:", results[-1]["train_rmse"],
      "Test:", results[-1]["test_rmse"])

AdaBoost best params: {'learning_rate': 1.0, 'n_estimators': 200}
AdaBoost RMSE - Train: 364.06999645994495 Test: 364.663986999746


### 5.5 Gradient Boosting

In [29]:
gbr = GradientBoostingRegressor(random_state=RANDOM_STATE)

param_grid_gbr = {
    "n_estimators": [100],
    "learning_rate": [0.05, 0.1],
    "max_depth": [2, 3],
    "max_features": ["sqrt"],
    "subsample": [0.8, 1.0]
}

grid_gbr = GridSearchCV(
    gbr, param_grid_gbr, scoring="neg_root_mean_squared_error", cv=5, n_jobs=-1
)
grid_gbr.fit(X_train_pc, y_train)
best_gbr = grid_gbr.best_estimator_

results.append({
    "model": "GradientBoosting",
    "train_rmse": rmse(y_train, best_gbr.predict(X_train_pc)),
    "test_rmse": rmse(y_test, best_gbr.predict(X_test_pc))
})
best_models["GradientBoosting"] = best_gbr

print("Gradient Boosting best params:", grid_gbr.best_params_)
print("Gradient Boosting RMSE - Train:", results[-1]["train_rmse"],
      "Test:", results[-1]["test_rmse"])

Gradient Boosting best params: {'learning_rate': 0.1, 'max_depth': 3, 'max_features': 'sqrt', 'n_estimators': 100, 'subsample': 0.8}
Gradient Boosting RMSE - Train: 285.33490243451837 Test: 290.935700053637


## 6. Model Evaluation Summary
<hr>

**Purpose:**  
Compare all tested models using RMSE.

**Insight:**  
Sorting by RMSE highlights the tradeoffs among linear, tree-based, and ensemble models.

In [30]:
results_df = pd.DataFrame(results)
results_df_sorted = results_df.sort_values("train_rmse")
print(results_df_sorted)

                 model  train_rmse   test_rmse
6         RandomForest  108.470773  292.112595
9     GradientBoosting  285.334902  290.935700
5      VotingRegressor  293.159122  309.960733
4  DecisionTree_pruned  298.479792  332.383209
0     LinearRegression  315.102008  315.214775
2                Ridge  315.102013  315.215436
1                Lasso  315.102138  315.216816
7              Bagging  334.315397  337.553360
8             AdaBoost  364.069996  364.663987
3  DecisionTree_simple  403.549932  407.874376


## 7. Predict on Scoring Data with Best PCA Model
<hr>

**Purpose:**  
- Refit model on full training set  
- Predict for scoring dataset  
- Create submission CSV  

**Note:**  
Best PCA-based model here differed from the best overall model in final submission.

In [32]:
# Predict Scoring Data with Best Model
best_model_name = results_df_sorted.iloc[0]["model"]
print("Best model based on test RMSE:", best_model_name)

best_model = best_models[best_model_name]

Best model based on test RMSE: RandomForest


In [33]:
# Refit PCA & scaler on full dataset
X_all_scaled = scaler.transform(X_analysis)
X_all_pc = pca.transform(X_all_scaled)

best_model.fit(X_all_pc, y)
scoring_preds = best_model.predict(X_scoring_pc)

In [34]:
# Build submission
submission = pd.DataFrame({
    "customer_id": scoring_data[id_col],
    "monthly_spend": scoring_preds
})

In [35]:
submission.to_csv("submission3.csv", index=False)
print("Submission file 'submission.csv' saved.")
submission.head()

Submission file 'submission.csv' saved.


,customer_id,monthly_spend
0,20451981,1684.89590
1,23251656,2601.67455
2,64082260,1398.73735
3,82505923,1944.36255
4,89981994,1497.20380


## 9. Missteps that Informed the Final Model
<hr>

**What Went Wrong**
> 1. Principal components were harder to interpret and offered no improvement in RMSE.
> 
> 2. Tree-Based Models failed to capture spending patterns -> Spending behavior exhibited mostly linear relationships, making trees unsuitable.
> 
> 3. Random Forests & Bagging Severely Overfit: increasing estimators or depth tuning did not solve the variance issue.
>
> 4. Limited feature engineering negatively impacted prediction performance.

**How These Missteps Shaped Notebook A: The failures in Notebook B directly motivated the final modeling decisions**  

| Misstep Identified                | Resulting Insight for Notebook A                         |
| --------------------------------- | -------------------------------------------------------- |
| PCA removed important structure   | Stick with engineered behavioral features instead of PCs |
| Trees & ensembles underperformed  | Spend prediction is mostly linear, not highly nonlinear - focus on linear models  |
| RF overfit badly                  | Model simplicity improves generalization                 |
| Lack of Feature Engineering    | Create ratios and interactions that capture valuable information              |

